# Exploratory Data Analysis (EDA) - Bot Detection Dataset
## Twitter Bot Detection Project

Cette notebook effectue une analyse exploratoire complète du dataset bot_detection_data.csv

## 1. Importation des libraires nécessaires

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuration de style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2. Chargement du dataset

In [2]:
# Chargement du dataset brut
df = pd.read_csv('../bot_detection_data.csv')

print(f'Nombre de lignes   : {df.shape[0]}')
print(f'Nombre de colonnes : {df.shape[1]}')
print('\n→ Le dataset contient 50 000 tweets annotés avec 11 attributs chacun, ce qui constitue un volume suffisant pour un entraînement ML fiable.')


Nombre de lignes   : 50000
Nombre de colonnes : 11

→ Le dataset contient 50 000 tweets annotés avec 11 attributs chacun, ce qui constitue un volume suffisant pour un entraînement ML fiable.


## 3. Analyse de la structure du dataset

In [3]:
# Aperçu des premières lignes du dataset
print(df.head())
print('\n→ On observe que les colonnes couvrent à la fois des données comportementales (Retweet Count, Mention Count) et des métadonnées de compte (Verified, Created At), utiles pour la détection.')


   User ID        Username  ...           Created At            Hashtags
0   132131           flong  ...  2020-05-11 15:29:50                 NaN
1   289683  hinesstephanie  ...  2022-11-26 05:18:10           both live
2   779715      roberttran  ...  2022-08-08 03:16:54         phone ahead
3   696168          pmason  ...  2021-08-14 22:27:05  ever quickly new I
4   704441          noah87  ...  2020-04-13 21:24:21     foreign mention

[5 rows x 11 columns]

→ On observe que les colonnes couvrent à la fois des données comportementales (Retweet Count, Mention Count) et des métadonnées de compte (Verified, Created At), utiles pour la détection.


In [4]:
# Informations générales sur les types et la complétude des colonnes
df.info()
print('\n→ Toutes les colonnes sauf Hashtags sont complètes (0 valeurs manquantes). La colonne Hashtags présente quelques absences qui seront traitées dans le pipeline.')


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   User ID         50000 non-null  int64
 1   Username        50000 non-null  str  
 2   Tweet           50000 non-null  str  
 3   Retweet Count   50000 non-null  int64
 4   Mention Count   50000 non-null  int64
 5   Follower Count  50000 non-null  int64
 6   Verified        50000 non-null  bool 
 7   Bot Label       50000 non-null  int64
 8   Location        50000 non-null  str  
 9   Created At      50000 non-null  str  
 10  Hashtags        41659 non-null  str  
dtypes: bool(1), int64(5), str(5)
memory usage: 9.5 MB

→ Toutes les colonnes sauf Hashtags sont complètes (0 valeurs manquantes). La colonne Hashtags présente quelques absences qui seront traitées dans le pipeline.


In [5]:
# Liste des colonnes disponibles
print("Colonnes du dataset :")
print(df.columns.tolist())
print("\n→ Les 11 colonnes disponibles permettent d'extraire des features comportementales et temporelles distinctives entre bots et utilisateurs humains.")


Colonnes du dataset :
['User ID', 'Username', 'Tweet', 'Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'Bot Label', 'Location', 'Created At', 'Hashtags']

→ Les 11 colonnes disponibles permettent d'extraire des features comportementales et temporelles distinctives entre bots et utilisateurs humains.


## 4. Analyse des valeurs manquantes

In [6]:
# Analyse des valeurs manquantes colonne par colonne
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Colonne': missing_values.index,
    'Valeurs manquantes': missing_values.values,
    'Pourcentage (%)': missing_percent.round(3).values
})

result = missing_df[missing_df['Valeurs manquantes'] > 0]
if len(result) == 0:
    print('Aucune valeur manquante détectée.')
else:
    print(result.to_string(index=False))
    print('\n→ Seule la colonne Hashtags présente des valeurs manquantes (16.68%), ce qui est attendu car tous les tweets ne contiennent pas de hashtag. Ces absences seront remplacées par une chaîne vide lors du nettoyage.')


 Colonne  Valeurs manquantes  Pourcentage (%)
Hashtags                8341           16.682

→ Seule la colonne Hashtags présente des valeurs manquantes (16.68%), ce qui est attendu car tous les tweets ne contiennent pas de hashtag. Ces absences seront remplacées par une chaîne vide lors du nettoyage.


## 5. Analyse de la variable cible (distribution des classes)

In [7]:
# Identification automatique de la colonne cible dans le dataset
possible_targets = ['Bot Label', 'bot', 'label', 'target', 'class', 'is_bot', 'Bot']
target_col = None

for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    target_col = df.columns[-1]

print(f'Colonne cible identifiée : {target_col}')
print(f'Valeurs possibles        : {sorted(df[target_col].unique().tolist())}')
print('\n→ La variable cible est binaire : 0 pour un utilisateur humain légitime, 1 pour un bot. Ce type de problème est une classification binaire supervisée.')


Colonne cible identifiée : Bot Label
Valeurs possibles        : [0, 1]

→ La variable cible est binaire : 0 pour un utilisateur humain légitime, 1 pour un bot. Ce type de problème est une classification binaire supervisée.


In [8]:
# Distribution des classes cibles (équilibre Bots/Humains)
class_dist = df[target_col].value_counts()
class_pct  = df[target_col].value_counts(normalize=True) * 100

print('Comptes absolus :')
print(f'  Bots (1)    : {class_dist.get(1, 0)}')
print(f'  Humains (0) : {class_dist.get(0, 0)}')
print(f'\nRépartition (%) :')
print(f'  Bots    : {class_pct.get(1, 0):.2f}%')
print(f'  Humains : {class_pct.get(0, 0):.2f}%')
print('\n→ Le dataset est parfaitement équilibré (~50/50). Aucun rééquilibrage (SMOTE, undersampling) ne sera nécessaire, ce qui simplifie le pipeline de modélisation.')


Comptes absolus :
  Bots (1)    : 25018
  Humains (0) : 24982

Répartition (%) :
  Bots    : 50.04%
  Humains : 49.96%

→ Le dataset est parfaitement équilibré (~50/50). Aucun rééquilibrage (SMOTE, undersampling) ne sera nécessaire, ce qui simplifie le pipeline de modélisation.


## 6. Statistiques descriptives

In [9]:
# Types de données de chaque colonne
print(df.dtypes)
print('\n→ Les colonnes numériques (int64) sont directement exploitables pour le ML. La colonne Verified (bool) sera convertie en entier. Les colonnes textuelles (str) seront traitées via feature engineering.')


User ID           int64
Username            str
Tweet               str
Retweet Count     int64
Mention Count     int64
Follower Count    int64
Verified           bool
Bot Label         int64
Location            str
Created At          str
Hashtags            str
dtype: object

→ Les colonnes numériques (int64) sont directement exploitables pour le ML. La colonne Verified (bool) sera convertie en entier. Les colonnes textuelles (str) seront traitées via feature engineering.


In [10]:
# Sélection des colonnes numériques brutes (hors variable cible)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if target_col in numeric_cols:
    numeric_cols.remove(target_col)

print(f'Colonnes numériques ({len(numeric_cols)} au total) :')
print(numeric_cols)
print('\n→ Ces 4 colonnes numériques brutes serviront de base au feature engineering : des ratios et scores composites en seront dérivés pour mieux discriminer les bots.')


Colonnes numériques (4 au total) :
['User ID', 'Retweet Count', 'Mention Count', 'Follower Count']

→ Ces 4 colonnes numériques brutes serviront de base au feature engineering : des ratios et scores composites en seront dérivés pour mieux discriminer les bots.


In [11]:
# Calcul des corrélations avec la variable cible Bot Label
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

if target_col in numeric_df.columns:
    target_corr = correlation_matrix[target_col].sort_values(ascending=False)
    print('Corrélations avec Bot Label (de la plus forte à la plus faible) :')
    print(target_corr.drop(target_col).to_string())
    print("\n→ Les corrélations brutes sont faibles (< 0.01), ce qui montre que les features simples ne suffisent pas. C'est précisément pourquoi nous construisons des features composites et graphiques pour capturer des signaux comportementaux plus subtils.")

else:
    print(correlation_matrix.iloc[:5, :5])


Corrélations avec Bot Label (de la plus forte à la plus faible) :
User ID           0.006059
Retweet Count     0.001250
Follower Count    0.001162
Mention Count    -0.006912

→ Les corrélations brutes sont faibles (< 0.01), ce qui montre que les features simples ne suffisent pas. C'est précisément pourquoi nous construisons des features composites et graphiques pour capturer des signaux comportementaux plus subtils.


## 10. Analyse des variables catégorielles

In [12]:
# Identification et résumé des colonnes catégorielles
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()

print(f'Colonnes catégorielles ({len(categorical_cols)} au total) :')
print(categorical_cols)

# Affichage du nombre de valeurs uniques par colonne (sans lister toutes les valeurs)
print('\nNombre de valeurs uniques par colonne :')
for col in categorical_cols:
    print(f'  {col:<15} : {df[col].nunique():>6} valeurs uniques')

print("\n→ La colonne Tweet possède autant de valeurs uniques que de lignes : chaque tweet est distinct. La colonne Location présente une forte cardinalité (~25 000 lieux), ce qui confirme qu'une extraction de features est préférable à un encodage direct.")



Colonnes catégorielles (5 au total) :
['Username', 'Tweet', 'Location', 'Created At', 'Hashtags']

Nombre de valeurs uniques par colonne :
  Username        :  40566 valeurs uniques
  Tweet           :  50000 valeurs uniques
  Location        :  25199 valeurs uniques
  Created At      :  49989 valeurs uniques
  Hashtags        :  34247 valeurs uniques

→ La colonne Tweet possède autant de valeurs uniques que de lignes : chaque tweet est distinct. La colonne Location présente une forte cardinalité (~25 000 lieux), ce qui confirme qu'une extraction de features est préférable à un encodage direct.


## 11. Résumé et Insights Clés

In [13]:
# Identification et résumé des colonnes catégorielles
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()

print(f'Colonnes catégorielles ({len(categorical_cols)} au total) :')
print(categorical_cols)

# Affichage du nombre de valeurs uniques par colonne (sans lister toutes les valeurs)
print('\nNombre de valeurs uniques par colonne :')
for col in categorical_cols:
    print(f'  {col:<15} : {df[col].nunique():>6} valeurs uniques')

print("\n→ La colonne Tweet possède autant de valeurs uniques que de lignes : chaque tweet est distinct. La colonne Location présente une forte cardinalité (~25 000 lieux), ce qui confirme qu'une extraction de features est préférable à un encodage direct.")



Colonnes catégorielles (5 au total) :
['Username', 'Tweet', 'Location', 'Created At', 'Hashtags']

Nombre de valeurs uniques par colonne :
  Username        :  40566 valeurs uniques
  Tweet           :  50000 valeurs uniques
  Location        :  25199 valeurs uniques
  Created At      :  49989 valeurs uniques
  Hashtags        :  34247 valeurs uniques

→ La colonne Tweet possède autant de valeurs uniques que de lignes : chaque tweet est distinct. La colonne Location présente une forte cardinalité (~25 000 lieux), ce qui confirme qu'une extraction de features est préférable à un encodage direct.
